Reference: https://github.com/drphilmarshall/SharPy/blob/main/sharpy/examples/inference.ipynb

In [ ]:
import numpy as np
import requests
import io
import importlib
import matplotlib.pyplot as plt
import sys
sys.path.append("../../")  # where the SharPy package lives
from sharpy.enhancement import Enhancement
from astropy.visualization import LinearStretch, AsinhStretch, LogStretch, ImageNormalize
import matplotlib.colors as colors


In [ ]:
def normalize_band(image, asinh_a=0.002):
    
    data = image

    vmin = -0.03
    #vmax = 200
    vmax = 10000

    norm = ImageNormalize(vmin=vmin, vmax=vmax,
                          #stretch=LinearStretch(),
                          #stretch=AsinhStretch(a=asinh_a),
                          stretch=LogStretch(a=10),
                          clip=True,
                         )

    scaled = norm(data)
    return scaled


def combine_RGB(R_image, G_image, B_image):
    
    R_channel = normalize_band(R_image)
    G_channel = normalize_band(G_image)
    B_channel = normalize_band(B_image)

    RGB_image = np.dstack([R_channel, G_channel, B_channel])

    return RGB_image

    

### Initialize the inference class with (pre-trained) RIM-Model provided by the SharPy package and ask for the 'GPU' execution option 

In [ ]:
my_enh = Enhancement(device='GPU', trained_model_file=None)

### Load data object and psf images and deconvolve & denoise the whole batch.

In [ ]:

# numpy arrays with shape: (num_objects, num_bands, num_pixels_i, num_pixels_j) 

BANDS = ('u', 'g', 'r', 'i', 'z', 'y')
observed_object_image = []
observed_psf_image = []

for band in BANDS:
    try:
        image = np.load(f'../../../cutout_{band}.npz')['image']
        psf = np.load(f'../../../cutout_{band}.npz')['psf']
    except:
        print(f'Missing {band}!')
        continue

    image = image[12:-11,12:-11]
    psf = psf[2:-1,2:-1]

    #image = image[11:-12,11:-12]
    #psf = psf[1:-2,1:-2]

    observed_object_image.append(image)
    observed_psf_image.append(psf)
    
#observed_object_image.insert(0,observed_object_image[0])
#observed_psf_image.insert(0,observed_psf_image[0])

observed_object_images = np.stack(observed_object_image)
observed_psf_images = np.stack(observed_psf_image)

print(np.shape(observed_object_images))
print(np.shape(observed_psf_images))



predicted_object_images, predicted_psf_images = my_enh.enhance(
    observed_object_images, 
    observed_psf_images
)


In [ ]:
print(np.shape(predicted_object_images))

In [ ]:
#predicted_object_images[0][0]

In [ ]:
fig, axs = plt.subplots(2, 6, figsize=(12,4))

for i, ax in enumerate(axs[0,:]):
    im = ax.imshow(observed_object_images[i], origin='lower')
    #fig.colorbar(im, ax=ax)
    ax.set_title(BANDS[i])
    ax.set_axis_off()
    
for i, ax in enumerate(axs[1,:]):
    
    im = ax.imshow(predicted_object_images[0][i], origin='lower',
                   norm=colors.LogNorm(vmin=1,vmax=10000)
                  )
    fig.colorbar(im, ax=ax)
    #ax.set_axis_off()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8,4))

B_image = observed_object_images[1]
G_image = observed_object_images[2]
R_image = observed_object_images[3]
RGB = combine_RGB(R_image, G_image, B_image)
axs[0].imshow(RGB, origin='lower')

B_image = predicted_object_images[0][1]
G_image = predicted_object_images[0][2]
R_image = predicted_object_images[0][3]
RGB = combine_RGB(R_image, G_image, B_image)
axs[1].imshow(RGB, origin='lower')

for ax in axs:
    ax.set_axis_off()